## TemPEST: Temporal Planning via SMT

**TemPEST** (Temporal Planner via Encoding into Satisfiability Testing) takes a temporal planning
problem and *encodes it into an SMT/OMT formula*: actions, fluents, durations and timing
constraints become logical and arithmetic constraints, and an off-the-shelf SMT solver
(Z3, MathSAT, OptiMathSAT) searches for a satisfying assignment. A model of that formula is then
decoded back into a temporal plan. It integrates as a planning engine into the
[Unified Planning](https://github.com/aiplan4eu/unified-planning) (UP) framework, so you build
problems with the familiar UP API and call TemPEST through `OneshotPlanner` / `AnytimePlanner`.

TemPEST comes in two flavours:

- **Satisficing** — find *any* valid plan, as fast as possible. The engine searches at increasing
  *horizons* `h = 2, 3, 4, …` (the number of plan steps) until the formula becomes satisfiable.
  In *anytime* mode it keeps going, blocking each plan found and yielding new ones.
- **Optimal** — find a plan that *minimises a quality metric* (makespan or summed action costs)
  using OMT (Optimization Modulo Theory).

The point of this notebook is **expressiveness**: because time is a first-class theory object in
the encoding, several features that are awkward (or impossible) for heuristic search planners fall
out naturally — *required concurrency*, *intermediate conditions and effects*, *timed goals*,
*continuous control parameters* the solver synthesises, and *exhaustive enumeration* of distinct
plans.

The walkthrough, in order: **installation** → a minimal *counter* problem with **anytime** plan
enumeration → **matchcellar** (required concurrency and timed goals) → **painter** (intermediate
conditions and effects) → **optimal makespan** → **action-cost** optimisation → **control
parameters** (continuous action parameters the solver synthesises) → a tour of the underlying
**SMT solvers and encodings** → a wrap-up of the available engine variants.

The reader is assumed to know temporal planning and Unified Planning, but not necessarily the
advanced features (optimality metrics, timed goals, control parameters) — those are introduced
gently.

## Installation

TemPEST needs the Unified Planning framework, the PySMT bindings (listed in `requirements.txt`),
and at least one SMT solver installed through PySMT. From the repository root:

```bash
pip install --pre unified-planning   # the planning framework (pre-release pinned by CI)
pip install -r requirements.txt      # PySMT and friends
pysmt-install --z3 --optimsat --confirm-agreement   # the SMT/OMT solvers
pip install plotly pandas            # for the plan-timeline plots used in this notebook
```

- **Z3** is the default OMT solver.
- **OptiMathSAT** (`optimsat`) adds the OMT capabilities to MathSAT.
- **plotly** (with **pandas**) backs `unified_planning.plot.plot_time_triggered_plan`, which we use
  throughout to draw each plan as a timeline.

After installing, run the notebook from the `doc/` directory (the next cell adds the repository
root to `sys.path` so that `import tempest` works in development mode).

In [1]:
import sys
sys.path.append("../") # go to parent directory to import modules

In [2]:
import sys, time, warnings
from unified_planning.shortcuts import (
    Problem, DurativeAction, InstantaneousAction,
    OneshotPlanner, AnytimePlanner,
    EndTiming, StartTiming, OpenTimeInterval, TimePointInterval, GlobalStartTiming,
    Fluent, Object,
    IntType, BoolType, RealType, UserType, Int,
    Equals, Not, And, GE, MinimizeMakespan, MinimizeActionCosts,
    get_environment,
)
from unified_planning.plot import plot_time_triggered_plan  # timeline visualisation of plans
warnings.filterwarnings("ignore")  # suppress quality-metric / support-check compatibility warnings

In [3]:
env = get_environment()
env.factory.add_engine("tempest",     "tempest.engine", "TempestEngine")
env.factory.add_engine("tempest-opt", "tempest.engine", "TempestOptimal")
env.credits_stream = None  # suppress engine credits banner
print("Registered: tempest, tempest-opt")

Registered: tempest, tempest-opt


## Problem 1: Counter — a minimal "hello world", solved exhaustively

The smallest interesting problem. An integer fluent `counter` starts at 0 and two **instantaneous**
actions act on it: `increase` adds 2, `decrease` subtracts 1. The goal is `counter == 3`.

The `+2` step size is deliberate: you *cannot* land on the odd value 3 using increases alone, so
every valid plan must include at least one `decrease` to fine-tune. The shortest plan therefore
uses three actions — two increases and one decrease — and there are exactly **three distinct
orderings** of them, each reaching the goal by a different route:

| Plan | counter trajectory |
|------|--------------------|
| increase, increase, decrease | 0 → 2 → 4 → 3 |
| increase, decrease, increase | 0 → 2 → 1 → 3 |
| decrease, increase, increase | 0 → -1 → 1 → 3 |

Because the actions are instantaneous, the problem has no continuous time: TemPEST returns a
`SequentialPlan` (a plain ordered list of actions, no timestamps). We use this problem twice in
this section: first to see the engine run end to end, then — in **anytime** mode — to enumerate
all three distinct plans, which differ in *which actions are chosen and in what order*, not merely
in timing.

In [4]:
counter_prob = Problem("counter")
counter = counter_prob.add_fluent("counter", IntType(), default_initial_value=0)

increase = InstantaneousAction("increase")
increase.add_increase_effect(counter, 2)

decrease = InstantaneousAction("decrease")
decrease.add_decrease_effect(counter, 1)

counter_prob.add_actions([increase, decrease])
counter_prob.add_goal(Equals(counter, 3))

print(counter_prob)

problem name = counter

fluents = [
  integer counter
]

actions = [
  action increase {
    preconditions = [
    ]
    effects = [
      counter += 2
    ]
  }
  action decrease {
    preconditions = [
    ]
    effects = [
      counter -= 1
    ]
  }
]

initial fluents default = [
  integer counter := 0
]

initial values = [
]

goals = [
  (counter == 3)
]




### Basic solve: incremental encoding + Z3

`OneshotPlanner` returns the first satisficing plan found. `output_stream` exposes the per-step search progress.

In [5]:
with OneshotPlanner(name="tempest", params={"incremental": True, "solver_name": "z3"}) as p:
    result = p.solve(counter_prob, output_stream=sys.stdout)

print("\nStatus:", result.status)
print("Plan:", result.plan)
# Instantaneous-only problem => SequentialPlan (no timestamps), so there is no
# timeline to draw here; the matchcellar problem below shows the timeline view.

No solution with bound 2. Elapsed_time: 0.016 seconds
No solution with bound 3. Elapsed_time: 0.017 seconds

Status: PlanGenerationResultStatus.SOLVED_SATISFICING
Plan: SequentialPlan:
    increase
    decrease
    increase


### Anytime mode: enumerating all the plans

`TempestEngine` implements UP's `AnytimePlannerMixin`: after finding a plan it adds a *blocking
clause* (the negation of that solution) to the solver and keeps searching, so `get_solutions`
yields a stream of distinct plans rather than stopping at the first. This is useful for offering a
user alternatives, or for downstream filtering by a criterion the planner does not optimise
directly.

The blocking clause negates the plan's *action selection* — which action fires at each step. The
enumerated plans therefore differ in **which actions are chosen and in what order**, not merely in
timing. There are exactly three shortest ways to reach `counter == 3` — the three orderings of two
`increase`s and one `decrease` — and the engine enumerates all of them. We cap the enumeration at
`MAX_PLANS`.

In [6]:
MAX_PLANS = 3

def trajectory(plan):
    """The value of `counter` after each action — to show same goal, different routes."""
    v, out = 0, [0]
    for a in plan.actions:
        v += 2 if a.action.name == "increase" else -1
        out.append(v)
    return out

with AnytimePlanner(name="tempest", params={"solver_name": "z3"}) as p:
    for i, result in enumerate(p.get_solutions(counter_prob, timeout=15)):
        print(f"Plan {i+1}: {result.plan}  counter: {trajectory(result.plan)}")
        if i + 1 >= MAX_PLANS:
            break

Plan 1: SequentialPlan:
    increase
    decrease
    increase  counter: [0, 2, 1, 3]
Plan 2: SequentialPlan:
    decrease
    increase
    increase  counter: [0, -1, 1, 3]
Plan 3: SequentialPlan:
    increase
    increase
    decrease  counter: [0, 2, 4, 3]


## Problem 2: Matchcellar — required concurrency

The classic *matchcellar* domain: a fuse can only be mended **while** a match is burning. The two
durative actions are:

- `light_match(m)` (duration 8) — `light` becomes true at the start of the burn and false again at
  its end; each match can be used only once;
- `mend_fuse(f)` (duration 5) — requires `light` to hold over the **whole open interval** of the
  action (`OpenTimeInterval(StartTiming(), EndTiming())`), and occupies your only free hand
  (`handfree`), so fuses are mended one at a time.

No valid plan can sequence these actions: `mend_fuse` **must execute strictly inside**
`light_match`. This *required concurrency* is the standard litmus test separating truly temporal
planners from those that reduce to sequential planning with durations — and in the SMT encoding it
needs no special machinery: the over-all condition simply becomes arithmetic constraints linking
the start/end time variables of the two actions.

The actions are **parameterised over user types** (`Match`, `Fuse`): TemPEST grounds each
object-typed parameter into an SMT symbol constrained to range over the finite set of objects of
the right type — the solver effectively *chooses which match and which fuse* to plug in at each
step.

In [7]:
mc_prob = Problem("matchcellar")
Match = UserType("Match")
Fuse  = UserType("Fuse")

light    = mc_prob.add_fluent("light",    BoolType(), default_initial_value=False)
handfree = mc_prob.add_fluent("handfree", BoolType(), default_initial_value=True)
used     = mc_prob.add_fluent("used",     BoolType(), m=Match, default_initial_value=False)
mended   = mc_prob.add_fluent("mended",   BoolType(), f=Fuse,  default_initial_value=False)

light_match = DurativeAction("light_match", m=Match)
light_match.set_fixed_duration(8)
light_match.add_condition(StartTiming(), Not(used(light_match.m)))
light_match.add_effect(StartTiming(), used(light_match.m), True)
light_match.add_effect(StartTiming(), light, True)
light_match.add_effect(EndTiming(),   light, False)

mend_fuse = DurativeAction("mend_fuse", f=Fuse)
mend_fuse.set_fixed_duration(5)
mend_fuse.add_condition(StartTiming(), handfree)
mend_fuse.add_condition(OpenTimeInterval(StartTiming(), EndTiming()), light)  # light DURING the mend
mend_fuse.add_effect(StartTiming(), handfree, False)
mend_fuse.add_effect(EndTiming(),   mended(mend_fuse.f), True)
mend_fuse.add_effect(EndTiming(),   handfree, True)

mc_prob.add_actions([light_match, mend_fuse])
mc_prob.add_objects([Object(f"m{i}", Match) for i in range(2)])
fuses = [mc_prob.add_object(f"f{i}", Fuse) for i in range(2)]
for f in fuses:
    mc_prob.add_goal(mended(f))

print(mc_prob)

problem name = matchcellar

types = [Match, Fuse]

fluents = [
  bool light
  bool handfree
  bool used[m=Match]
  bool mended[f=Fuse]
]

actions = [
  durative action light_match(Match m) {
    duration = [8, 8]
    conditions = [
      [start]:
        (not used(m))
    ]
    effects = [
      start:
        used(m) := true:
        light := true:
      end:
        light := false:
    ]
    simulated effects = [
    ]
  }
  durative action mend_fuse(Fuse f) {
    duration = [5, 5]
    conditions = [
      [start]:
        handfree
      (start, end):
        light
    ]
    effects = [
      start:
        handfree := false:
      end:
        mended(f) := true:
        handfree := true:
    ]
    simulated effects = [
    ]
  }
]

objects = [
  Match: [m0, m1]
  Fuse: [f0, f1]
]

initial fluents default = [
  bool light := false
  bool handfree := true
  bool used[m=Match] := false
  bool mended[f=Fuse] := false
]

initial values = [
]

goals = [
  mended(f0)
  mended(f1)
]




In [8]:
with OneshotPlanner(name="tempest", params={"solver_name": "z3"}) as p:
    mc_result = p.solve(mc_prob, output_stream=sys.stdout)

print("\nPlan:")
for start, action, duration in mc_result.plan.timed_actions:
    print(f"  t={float(start):.2f}  {action}  [dur={duration}]")

# The timeline makes the required concurrency visually obvious:
# each mend_fuse bar sits strictly inside a light_match bar.
plot_time_triggered_plan(mc_result.plan)

No solution with bound 2. Elapsed_time: 0.009 seconds
No solution with bound 3. Elapsed_time: 0.015 seconds
No solution with bound 4. Elapsed_time: 0.022 seconds
No solution with bound 5. Elapsed_time: 0.030 seconds
No solution with bound 6. Elapsed_time: 0.042 seconds

Plan:
  t=0.00  light_match(m1)  [dur=8]
  t=0.00  mend_fuse(f1)  [dur=5]
  t=8.33  light_match(m0)  [dur=8]
  t=8.33  mend_fuse(f0)  [dur=5]


### Timed goals: adding a deadline

Time being a theory object also makes **timed goals** natural: instead of (or in addition to)
requiring a goal to hold *at the end of the plan*, we can require it to hold **at a specific time
point**. Here we ask for both fuses to be mended *by time 16* —
`add_timed_goal(TimePointInterval(GlobalStartTiming(16)), …)` — so the whole repair must fit
inside a 16-time-unit window. In the encoding this is just one more constraint on the time
variables; no replanning machinery or special goal handling is involved.

Beware that the deadline directly shapes the combinatorics: the tighter the window, the more the
solver must interleave actions to meet it, and instances get harder quickly as the deadline
approaches the theoretical minimum.

In [9]:
mc_deadline = mc_prob.clone()
for f in fuses:
    # `mended` is never unset, so "holds at t=16" means "achieved by t=16": a deadline.
    mc_deadline.add_timed_goal(TimePointInterval(GlobalStartTiming(16)), mended(f))

with OneshotPlanner(name="tempest", params={"solver_name": "z3"}) as p:
    mc_dl_result = p.solve(mc_deadline, output_stream=sys.stdout)

print("\nPlan (everything mended by t=16):")
for start, action, duration in mc_dl_result.plan.timed_actions:
    print(f"  t={float(start):.2f}  {action}  [dur={duration}]")

plot_time_triggered_plan(mc_dl_result.plan)

No solution with bound 2. Elapsed_time: 0.009 seconds
No solution with bound 3. Elapsed_time: 0.014 seconds
No solution with bound 4. Elapsed_time: 0.021 seconds
No solution with bound 5. Elapsed_time: 0.031 seconds
No solution with bound 6. Elapsed_time: 0.043 seconds

Plan (everything mended by t=16):
  t=0.00  light_match(m0)  [dur=8]
  t=0.00  mend_fuse(f0)  [dur=5]
  t=8.75  light_match(m1)  [dur=8]
  t=8.75  mend_fuse(f1)  [dur=5]


## Problem 3: Painter — intermediate conditions and effects

PDDL durative actions only allow conditions and effects at the *start*, the *end*, or *over all*
of an action. TemPEST supports conditions and effects at **arbitrary intermediate time points** —
in the SMT encoding a timing like `StartTiming(4)` is simply `t_start + 4`, another linear term.

**The painter domain.** A set of `n_i` items must each undergo a fixed sequence of `n_t`
treatments (think: primer → paint → varnish). A single shared station applies one treatment at a
time (`busy`), and treatments on an item must be applied in order. The action `make_treatment`
(duration 15) uses intermediate timings throughout its execution:

- at **start**: the station becomes `busy` and the treatment is marked `started`;
- at **start + 4**: the item is `treated` and the station *frees up* — the next treatment can
  begin while this action is still running;
- at **start + 10**: the item becomes `ready` for the *next* treatment;
- at **end**: the *following* treatment must already have `started` — a condition that forces the
  pipeline to keep flowing.

The shared station plus the orderings make this a genuine scheduling problem whose difficulty
grows quickly with `n_t` and `n_i`. We will reuse it later as the workhorse for the solver tour.

In [10]:
def make_painter_problem(n_t, n_i):
    """A self-contained painter problem: n_i items, each needing n_t sequential treatments,
    sharing a single station. Returns a UP Problem (satisficing — no quality metric)."""
    Item      = UserType("Item")
    Treatment = UserType("Treatment")

    # Fluents
    busy        = Fluent("busy")                                    # station occupied?
    treated     = Fluent("treated",     BoolType(), i=Item, t=Treatment)
    started     = Fluent("started",     BoolType(), i=Item, t=Treatment)
    ready       = Fluent("ready",       BoolType(), i=Item, t=Treatment)
    counter     = Fluent("counter",     IntType(0, 31), t=Treatment)   # next item to get treatment t
    item_id     = Fluent("item_id",     IntType(0, 31), i=Item)        # item's position in the queue
    consecutive = Fluent("consecutive", BoolType(), t1=Treatment, t2=Treatment)

    # The durative action, with intermediate conditions and effects.
    mt = DurativeAction("make_treatment", i=Item, t=Treatment, next=Treatment)
    mt.set_fixed_duration(15)
    # Start conditions: it is this item's turn for treatment t, t is followed by `next`,
    # the station is free, t not already done/started on i, and i is ready for t.
    mt.add_condition(StartTiming(), Equals(item_id(mt.i), counter(mt.t)))
    mt.add_condition(StartTiming(), consecutive(mt.t, mt.next))
    mt.add_condition(StartTiming(), Not(busy))
    mt.add_condition(StartTiming(), Not(treated(mt.i, mt.t)))
    mt.add_condition(StartTiming(), Not(started(mt.i, mt.t)))
    mt.add_condition(StartTiming(), ready(mt.i, mt.t))
    # Start effects: advance the queue, occupy the station, mark as started.
    mt.add_increase_effect(StartTiming(), counter(mt.t), 1)
    mt.add_effect(StartTiming(), busy, True)
    mt.add_effect(StartTiming(), started(mt.i, mt.t), True)
    # Intermediate effects: at +4 the item is treated and the station frees up;
    # at +10 the item becomes ready for the next treatment.
    mt.add_effect(StartTiming(4),  treated(mt.i, mt.t), True)
    mt.add_effect(StartTiming(4),  busy, False)
    mt.add_effect(StartTiming(10), ready(mt.i, mt.next), True)
    # End condition: the item must already be on its way through the following treatment.
    mt.add_condition(EndTiming(), started(mt.i, mt.next))

    p = Problem(f"painter_{n_t}_{n_i}")
    for f in [busy, treated, started, ready, consecutive]:
        p.add_fluent(f, default_initial_value=False)
    for f in [counter, item_id]:
        p.add_fluent(f, default_initial_value=0)
    p.add_action(mt)

    # n_t real treatments t0..t(n_t-1), plus a sentinel t(n_t) used as the "next" of the last one.
    p.add_objects([Object(f"t{i}", Treatment) for i in range(n_t + 1)])
    p.add_objects([Object(f"i{i}", Item)      for i in range(n_i)])

    for i in range(n_i):
        p.set_initial_value(item_id(p.object(f"i{i}")), i)
    for t in range(n_t):
        p.set_initial_value(consecutive(p.object(f"t{t}"), p.object(f"t{t+1}")), True)
    for i in range(n_i):
        p.set_initial_value(started(p.object(f"i{i}"), p.object(f"t{n_t}")), True)  # sentinel
        p.set_initial_value(ready(p.object(f"i{i}"),   p.object("t0")), True)
    for i in range(n_i):
        for t in range(n_t):
            p.add_goal(treated(p.object(f"i{i}"), p.object(f"t{t}")))
    return p


painter_prob = make_painter_problem(3, 2)   # 2 items, 3 treatments each — solves in a couple of seconds
print(f"{painter_prob.name}: {len(painter_prob.actions)} action schema, "
      f"{len(list(painter_prob.all_objects))} objects, {len(painter_prob.goals)} goals")

painter_3_2: 1 action schema, 6 objects, 6 goals


In [11]:
with OneshotPlanner(name="tempest", params={"solver_name": "z3"}) as p:
    painter_result = p.solve(painter_prob)

print(f"Status: {painter_result.status.name}\n")
for start, action, duration in painter_result.plan.timed_actions:
    print(f"  t={float(start):6.2f}  {action}  [dur={duration}]")

# The timeline shows the station handover: each treatment starts as soon as the
# previous action frees the station at +4, well before that action has ended.
plot_time_triggered_plan(painter_result.plan)

Status: SOLVED_SATISFICING

  t=  0.00  make_treatment(i0, t0, t1)  [dur=15]
  t=  6.00  make_treatment(i1, t0, t1)  [dur=15]
  t= 12.00  make_treatment(i0, t1, t2)  [dur=15]
  t= 18.00  make_treatment(i1, t1, t2)  [dur=15]
  t= 24.00  make_treatment(i0, t2, t3)  [dur=15]
  t= 30.00  make_treatment(i1, t2, t3)  [dur=15]


## Optimal Planning: Minimizing Makespan

The engines so far returned *any* valid plan. The `tempest-opt` engine instead returns a
**provably optimal** one. It uses OMT (Optimization Modulo Theory): with `sat_before_opt=True` it
first finds a satisficing bound via plain SAT, then hands the formula to an OMT optimizer that
*minimises the objective* — here the **makespan**, the time at which the last action ends — and
proves no shorter plan exists at that horizon. (`ground_abstract_step=True` grounds the trailing
"abstract" step so the optimiser reasons over concrete actions.) OMT requires an OMT-capable
solver such as OptiMathSAT or Z3.

A subtle point this example makes: the optimal plan can contain **more** actions than the
satisficing one, yet achieve a shorter makespan by running them concurrently.

**Problem — build pipeline**  
Two compilation units (`compile_a`, `compile_b`, duration 2 each) are independent and
can overlap; a final `link` step (duration 1) requires both to have finished.
An alternative `monolithic_build` action (duration 5) does everything in one shot.

| Plan | Actions | Makespan |
|------|---------|----------|
| monolithic | 1 | 5 |
| compile\_a \|\| compile\_b → link | 3 | 3 |

A satisficing planner stops at the first valid plan it finds — here `monolithic_build`
(horizon 3, one action slot). The optimal planner proves that running the two compile
steps in parallel and then linking achieves makespan 3.

In [12]:
pipeline_prob = Problem("build_pipeline")
compiled_a = pipeline_prob.add_fluent("compiled_a", BoolType(), default_initial_value=False)
compiled_b = pipeline_prob.add_fluent("compiled_b", BoolType(), default_initial_value=False)
linked     = pipeline_prob.add_fluent("linked",     BoolType(), default_initial_value=False)

monolithic = DurativeAction("monolithic_build")
monolithic.set_fixed_duration(Int(5))
monolithic.add_effect(EndTiming(), linked, True)

compile_a = DurativeAction("compile_a")
compile_a.set_fixed_duration(Int(2))
compile_a.add_effect(EndTiming(), compiled_a, True)

compile_b = DurativeAction("compile_b")
compile_b.set_fixed_duration(Int(2))
compile_b.add_effect(EndTiming(), compiled_b, True)

link = DurativeAction("link")
link.set_fixed_duration(Int(1))
link.add_condition(StartTiming(), And(compiled_a, compiled_b))
link.add_effect(EndTiming(), linked, True)

pipeline_prob.add_actions([monolithic, compile_a, compile_b, link])
pipeline_prob.add_goal(linked)
pipeline_prob.add_quality_metric(MinimizeMakespan())

def plan_makespan(plan):
    return max(float(s) + float(d) for s, _, d in plan.timed_actions)

print(pipeline_prob)

problem name = build_pipeline

fluents = [
  bool compiled_a
  bool compiled_b
  bool linked
]

actions = [
  durative action monolithic_build {
    duration = [5, 5]
    conditions = [
    ]
    effects = [
      end:
        linked := true:
    ]
    simulated effects = [
    ]
  }
  durative action compile_a {
    duration = [2, 2]
    conditions = [
    ]
    effects = [
      end:
        compiled_a := true:
    ]
    simulated effects = [
    ]
  }
  durative action compile_b {
    duration = [2, 2]
    conditions = [
    ]
    effects = [
      end:
        compiled_b := true:
    ]
    simulated effects = [
    ]
  }
  durative action link {
    duration = [1, 1]
    conditions = [
      [start]:
        (compiled_a and compiled_b)
    ]
    effects = [
      end:
        linked := true:
    ]
    simulated effects = [
    ]
  }
]

initial fluents default = [
  bool compiled_a := false
  bool compiled_b := false
  bool linked := false
]

initial values = [
]

goals = [
  linked

In [13]:
with OneshotPlanner(name="tempest", params={"solver_name": "z3"}) as p:
    sat_pipeline = p.solve(pipeline_prob, output_stream=sys.stdout)

print(f"\nSatisficing plan  (makespan {plan_makespan(sat_pipeline.plan):.1f}):")
for start, action, dur in sat_pipeline.plan.timed_actions:
    print(f"  t={float(start):.2f}  {action}  [dur={dur}]")

No solution with bound 2. Elapsed_time: 0.008 seconds

Satisficing plan  (makespan 5.0):
  t=0.00  monolithic_build  [dur=5]


In [14]:
with OneshotPlanner(name="tempest-opt",
                    params={"solver_name": "z3", "sat_before_opt": True,
                            "ground_abstract_step": True}) as p:
    opt_pipeline = p.solve(pipeline_prob, output_stream=sys.stdout)

print(f"\nOptimal plan  (makespan {plan_makespan(opt_pipeline.plan):.1f}):")
for start, action, dur in opt_pipeline.plan.timed_actions:
    print(f"  t={float(start):.2f}  {action}  [dur={dur}]")

print(f"\nSatisficing makespan : {plan_makespan(sat_pipeline.plan):.1f}  ({len(sat_pipeline.plan.timed_actions)} action)")
print(f"Optimal makespan     : {plan_makespan(opt_pipeline.plan):.1f}  ({len(opt_pipeline.plan.timed_actions)} actions, parallel)")

# Visualise the optimal plan: compile_a and compile_b overlap, then link.
plot_time_triggered_plan(opt_pipeline.plan)

No SAT solution with bound 2. Elapsed_time: 0.008 seconds
SAT solution with bound 3. Elapsed_time: 0.013 seconds
Cost with bound 3: 603/200. Elapsed_time: 0.032 seconds
Cost with bound 4: 603/200. Elapsed_time: 0.060 seconds


OPT solution with bound 5: 301/100. Elapsed_time: 0.060 seconds

Optimal plan  (makespan 3.0):
  t=0.00  compile_a  [dur=2]
  t=0.00  compile_b  [dur=2]
  t=2.01  link  [dur=1]

Satisficing makespan : 5.0  (1 action)
Optimal makespan     : 3.0  (3 actions, parallel)


## Action Cost Optimization

`MinimizeActionCosts` assigns a numeric cost to each action; the planner minimises the
total cost accumulated over all executed actions, independently of time.

**Problem — relay delivery**  
A package must reach the customer. Two routes are available:

- **express** (duration 1, cost 90): a single premium-courier action.
- **economy route** (cost 30 total): three sequential steps —
  `to_hub` → `to_depot` → `deliver_local` — each costing 10 credits
  but taking 6 time units overall.

| Plan | Actions | Total cost | Makespan |
|------|---------|-----------|----------|
| express | 1 | 90 | 1 |
| to\_hub → to\_depot → deliver\_local | 3 | 30 | 6 |

The satisficing planner encounters `express` at the lowest feasible horizon and returns
it immediately. The optimal planner (`tempest-opt` with `MinimizeActionCosts`) proves
that the economy route has lower cost and returns it instead — a plan with **more**
actions but **lower** total cost.

In [15]:
relay_prob = Problem("relay_delivery")
delivered   = relay_prob.add_fluent("delivered",   BoolType(), default_initial_value=False)
at_hub      = relay_prob.add_fluent("at_hub",      BoolType(), default_initial_value=False)
at_depot    = relay_prob.add_fluent("at_depot",    BoolType(), default_initial_value=False)

express = DurativeAction("express")
express.set_fixed_duration(Int(1))
express.add_effect(EndTiming(), delivered, True)

to_hub = DurativeAction("to_hub")
to_hub.set_fixed_duration(Int(2))
to_hub.add_effect(EndTiming(), at_hub, True)

to_depot = DurativeAction("to_depot")
to_depot.set_fixed_duration(Int(2))
to_depot.add_condition(StartTiming(), at_hub)
to_depot.add_effect(EndTiming(), at_depot, True)

deliver_local = DurativeAction("deliver_local")
deliver_local.set_fixed_duration(Int(2))
deliver_local.add_condition(StartTiming(), at_depot)
deliver_local.add_effect(EndTiming(), delivered, True)

COSTS = {express: Int(90), to_hub: Int(10), to_depot: Int(10), deliver_local: Int(10)}
relay_prob.add_actions([express, to_hub, to_depot, deliver_local])
relay_prob.add_goal(delivered)
relay_prob.add_quality_metric(MinimizeActionCosts(COSTS))

COST_LOOKUP = {"express": 90, "to_hub": 10, "to_depot": 10, "deliver_local": 10}

def plan_action_cost(plan):
    return sum(COST_LOOKUP[a.action.name] for _, a, _ in plan.timed_actions)

print(relay_prob)

problem name = relay_delivery

fluents = [
  bool delivered
  bool at_hub
  bool at_depot
]

actions = [
  durative action express {
    duration = [1, 1]
    conditions = [
    ]
    effects = [
      end:
        delivered := true:
    ]
    simulated effects = [
    ]
  }
  durative action to_hub {
    duration = [2, 2]
    conditions = [
    ]
    effects = [
      end:
        at_hub := true:
    ]
    simulated effects = [
    ]
  }
  durative action to_depot {
    duration = [2, 2]
    conditions = [
      [start]:
        at_hub
    ]
    effects = [
      end:
        at_depot := true:
    ]
    simulated effects = [
    ]
  }
  durative action deliver_local {
    duration = [2, 2]
    conditions = [
      [start]:
        at_depot
    ]
    effects = [
      end:
        delivered := true:
    ]
    simulated effects = [
    ]
  }
]

initial fluents default = [
  bool delivered := false
  bool at_hub := false
  bool at_depot := false
]

initial values = [
]

goals = [
  deliv

In [16]:
with OneshotPlanner(name="tempest", params={"solver_name": "z3"}) as p:
    sat_relay = p.solve(relay_prob, output_stream=sys.stdout)

print(f"\nSatisficing plan  (total cost: {plan_action_cost(sat_relay.plan)}):")
for start, action, dur in sat_relay.plan.timed_actions:
    c = COST_LOOKUP[action.action.name]
    print(f"  t={float(start):.3f}  {action}  [dur={dur}  cost={c}]")

No solution with bound 2. Elapsed_time: 0.007 seconds

Satisficing plan  (total cost: 90):
  t=0.000  express  [dur=1  cost=90]


In [17]:
with OneshotPlanner(name="tempest-opt",
                    params={"solver_name": "z3", "sat_before_opt": True,
                            "ground_abstract_step": True}) as p:
    opt_relay = p.solve(relay_prob)  # no output_stream: internal weights are not plan costs

print(f"Optimal plan  (total cost: {plan_action_cost(opt_relay.plan)}):")
for start, action, dur in opt_relay.plan.timed_actions:
    c = COST_LOOKUP[action.action.name]
    print(f"  t={float(start):.3f}  {action}  [dur={dur}  cost={c}]")

print(f"\nSatisficing cost : {plan_action_cost(sat_relay.plan)}  ({len(sat_relay.plan.timed_actions)} action,  makespan {max(float(s)+float(d) for s,_,d in sat_relay.plan.timed_actions):.1f})")
print(f"Optimal cost     : {plan_action_cost(opt_relay.plan)}  ({len(opt_relay.plan.timed_actions)} actions, makespan {max(float(s)+float(d) for s,_,d in opt_relay.plan.timed_actions):.1f})")

# Visualise the optimal plan: the three-step economy route along the timeline.
plot_time_triggered_plan(opt_relay.plan)

Optimal plan  (total cost: 30):
  t=0.000  to_hub  [dur=2  cost=10]
  t=2.013  to_depot  [dur=2  cost=10]
  t=4.022  deliver_local  [dur=2  cost=10]

Satisficing cost : 90  (1 action,  makespan 1.0)
Optimal cost     : 30  (3 actions, makespan 6.0)


## Control Parameters: Continuous Action Parameters

Every action parameter we have seen so far ranged over a *finite* set — objects of a user type
(`Ball`, `Location`). TemPEST also supports **control parameters**: action parameters drawn from an
*infinite, continuous* domain (a real-valued type), whose value the **solver synthesises** as part
of planning rather than choosing from an enumerated list. This falls out naturally from the SMT
encoding — such a parameter simply becomes a *free real variable*, constrained to its declared
range, that the solver assigns while satisfying the action's conditions and effects.

This lets a single action stand in for a continuum of grounded actions. Instead of declaring
`set_temperature_100`, `set_temperature_150`, … you declare one `set_temperature(n_degrees)` with
`n_degrees : RealType(0, 200)` and let the solver compute exactly how many degrees are needed.

**Two practical caveats:**

1. Control parameters are supported by the **satisficing** `tempest` engine, **not** by the
   optimal `tempest-opt` engine.
2. A real-valued action parameter makes Unified Planning classify the problem as
   `GENERAL_NUMERIC_PLANNING`, which it cannot *automatically* verify TemPEST handles — so an
   `AnytimePlanner`/`OneshotPlanner` auto-selection would skip it. We therefore request the engine
   **by name** (`name="tempest"`); UP emits a harmless "cannot establish support" warning, which is
   already filtered out at the top of this notebook.

**Example — an oven.** Temperature starts at 20°. A single `set_temperature(n_degrees)` action
raises it by a synthesised amount; the goal is to reach at least 200°.

In [18]:
oven = Problem("oven")
temperature = oven.add_fluent("temperature", RealType(0, 500), default_initial_value=20)

# n_degrees is a CONTROL PARAMETER: a continuous value the solver will synthesise.
set_temperature = InstantaneousAction("set_temperature", n_degrees=RealType(0, 200))
set_temperature.add_increase_effect(temperature, set_temperature.n_degrees)
oven.add_action(set_temperature)
oven.add_goal(GE(temperature, 200))

with OneshotPlanner(name="tempest", params={"solver_name": "z3"}) as p:
    result = p.solve(oven)

print("Status:", result.status)
print("Plan:  ", result.plan)
print("\nThe solver SYNTHESISED n_degrees (it was not picked from a finite list):")
print("starting at 20°, it chose the increment needed to reach the 200° goal.")

Status: PlanGenerationResultStatus.SOLVED_SATISFICING
Plan:   SequentialPlan:
    set_temperature(180)

The solver SYNTHESISED n_degrees (it was not picked from a finite list):
starting at 20°, it chose the increment needed to reach the 200° goal.


Control parameters work on **durative** actions too. Here a `charge` action takes a fixed time
unit and, over that time, adds a synthesised `amount` to a battery; the goal is to reach 75%.

In [19]:
charger = Problem("charger")
charge_level = charger.add_fluent("charge", RealType(0, 100), default_initial_value=10)

charge_battery = DurativeAction("charge_battery", amount=RealType(0, 90))   # amount: control parameter
charge_battery.set_fixed_duration(1)
charge_battery.add_increase_effect(EndTiming(), charge_level, charge_battery.amount)
charger.add_action(charge_battery)
charger.add_goal(GE(charge_level, 75))

with OneshotPlanner(name="tempest", params={"solver_name": "z3"}) as p:
    result = p.solve(charger)

print("Status:", result.status)
print("Plan:  ", result.plan, "   (10% start + synthesised amount = 75%)")

Status: PlanGenerationResultStatus.SOLVED_SATISFICING
Plan:   TimeTriggeredPlan:
    0.0: charge_battery(65) [1.0]    (10% start + synthesised amount = 75%)


## Under the Hood: Solvers and Encodings

TemPEST is solver-agnostic: any PySMT-installed solver supporting QF_LRA can be plugged in via
`solver_name`. For *optimal* planning the solver must additionally support OMT — currently
**Z3** and **OptiMathSAT** (`optimsat`).

Orthogonally to the solver, TemPEST offers two encoding strategies for the horizon search
`h = 2, 3, 4, …`:

- **Incremental** (`incremental=True`, the default) — keeps a single solver alive across horizons.
  Going from `h` to `h+1` only *pushes new constraints* onto the existing solver, so all the
  reasoning the solver already did (learned clauses, theory lemmas) is reused.
- **Monolithic** (`incremental=False`) — rebuilds the *entire* formula from scratch at each
  horizon and hands it to a fresh solver. Simpler, but it throws away all previous work.

The cell below crosses both dimensions on the painter problem from earlier. Single-run wall-clock
times in a notebook are indicative, not a benchmark — but the incremental advantage widens as the
formulas grow (try `make_painter_problem(3, 3)` or `(4, 2)`).

In [20]:
# Solve the same painter problem with every solver x encoding combination.
print(f"{'solver':10s} {'encoding':12s} {'status':20s} {'time':>7s}")
for solver in ["z3", "optimsat"]:
    for label, incremental in [("incremental", True), ("monolithic", False)]:
        t0 = time.perf_counter()
        with OneshotPlanner(name="tempest",
                            params={"incremental": incremental, "solver_name": solver}) as p:
            res = p.solve(painter_prob)
        elapsed = time.perf_counter() - t0
        print(f"{solver:10s} {label:12s} {res.status.name:20s} {elapsed:6.2f}s")

solver     encoding     status                  time


z3         incremental  SOLVED_SATISFICING     1.82s


z3         monolithic   SOLVED_SATISFICING     2.98s


optimsat   incremental  SOLVED_SATISFICING     2.29s


optimsat   monolithic   SOLVED_SATISFICING     1.86s


## Wrap-Up

This notebook used two engines, but TemPEST ships seven variants, all combinations of the same
encoder and engine machinery (`tempest/engine.py`):

| Engine class | Objective | Notes |
|---|---|---|
| `TempestEngine` | satisficing | anytime: blocks solutions and continues |
| `TempestNonIncremental` | satisficing | monolithic encoding only |
| `TempestOptimal` | optimal (OMT) | makespan / action costs |
| `TempestOptimalNonIncremental` | optimal (OMT) | monolithic encoding only |
| `TempestLiftedAbstractStep` | optimal (OMT) | lifted (ungrounded) abstract step |
| `TempestOnlyOMT` | optimal (OMT) | skips the SAT phase, straight to OMT |
| `TempestLiftedAbstractStepOnlyOMT` | optimal (OMT) | lifted + OMT-only combined |

What we saw, in one line each:

- **Counter / anytime** — exhaustive enumeration of distinct plans via blocking clauses.
- **Matchcellar** — required concurrency and timed-goal deadlines, natural in the encoding.
- **Painter** — conditions and effects at arbitrary intermediate time points.
- **Build pipeline / relay delivery** — provably optimal makespan and action-cost plans via OMT.
- **Oven / charger** — continuous control parameters synthesised by the solver.

**Pointers**: the encoding is described formally in the paper draft under `paper/`; the
benchmark domains used in CI live in `benchmarks/test_cases/`; the engine and encoder sources
are under `tempest/`.